# Chapitre 5 - Cours 1 : polynômes par morceaux, splines et B-splines

Ce premier notebook du chapitre 5 (*Basis Expansions and Regularization*) présente
l'idée centrale du chapitre : **remplacer la variable d'entrée $X$ par des
transformations $h_1(X), \dots, h_M(X)$** puis ajuster un modèle *linéaire* dans ces
nouvelles variables. Nous verrons :

1. les expansions en bases de fonctions (§5.1) ;
2. les polynômes par morceaux et les splines (§5.2, figures 5.1 et 5.2) ;
3. la base des B-splines et la récurrence de Cox-de Boor (annexe du chapitre, figure 5.20) ;
4. les splines cubiques naturelles et la variance ponctuelle (§5.2.1, figure 5.3) ;
5. l'exemple de la reconnaissance de phonèmes : filtrer les entrées par une base de
   splines (§5.2.3 et §5.3, figure 5.5).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl.ch05_splines import (piecewise_polynomial_basis, truncated_power_basis,
                              bspline_basis, natural_spline_basis, logistic_irls)
set_style()

## 1. Expansions en bases de fonctions

Les modèles linéaires des chapitres 3 et 4 supposent que la fonction de régression
$f(X) = \mathbb E(Y \mid X)$ est linéaire en $X$. C'est rarement vrai, mais c'est une
approximation commode. Pour aller au-delà **sans abandonner les outils linéaires**, on
se donne des fonctions $h_m : \mathbb R^p \to \mathbb R$, $m = 1, \dots, M$, et l'on pose
(éq. (5.1) du livre)

$$
f(X) = \sum_{m=1}^{M} \beta_m\, h_m(X).
$$

Le modèle est **linéaire en les paramètres** $\beta_m$ : une fois les $h_m$ choisies,
l'ajustement par moindres carrés est exactement celui du chapitre 3, avec la matrice de
régression $\mathbf H = (h_m(x_i))_{i,m}$ de taille $N \times M$ :
$\hat\beta = (\mathbf H^T \mathbf H)^{-1}\mathbf H^T \mathbf y$.